# ESG Event-Driven Alpha Strategy - Quick Start

This notebook demonstrates the basic usage of the ESG Event-Driven Alpha Strategy.

## Setup

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Import project modules
from src.data import PriceFetcher, FamaFrenchFactors
from src.nlp import ESGEventDetector, FinancialSentimentAnalyzer, ReactionFeatureExtractor
from src.signals import ESGSignalGenerator, PortfolioConstructor
from src.backtest import BacktestEngine, PerformanceAnalyzer, FactorAnalysis

Install with: pip install transformers torch
Install with: pip install transformers torch


## Step 1: Fetch Price Data

In [ ]:
# Define parameters
tickers = ['AAPL', 'TSLA', 'XOM', 'JPM', 'MSFT']
start_date = '2023-01-01'
end_date = '2023-12-31'

# Fetch price data
price_fetcher = PriceFetcher()
prices = price_fetcher.fetch_price_data(tickers, start_date, end_date)

print(f"Fetched {len(prices)} rows of price data")
prices.head()

## Step 2: Detect ESG Events

In [ ]:
# Initialize event detector
event_detector = ESGEventDetector()

# Example ESG event texts
example_texts = [
    "AAPL announced major environmental fine from EPA for pollution violations.",
    "TSLA disclosed data breach affecting customer information.",
    "XOM facing discrimination lawsuit from employees."
]

# Detect events
for i, text in enumerate(example_texts):
    result = event_detector.detect_event(text)
    print(f"\nText {i+1}: {text[:60]}...")
    print(f"Event detected: {result['has_event']}")
    if result['has_event']:
        print(f"Category: {result['category_full']}")
        print(f"Confidence: {result['confidence']:.2f}")
        print(f"Keywords: {result['matched_keywords'][:3]}")

## Step 3: Analyze Sentiment

In [ ]:
# Initialize sentiment analyzer
sentiment_analyzer = FinancialSentimentAnalyzer()

# Example financial texts
financial_texts = [
    "Strong quarterly earnings beat expectations.",
    "Company facing major regulatory investigation.",
    "Product launch received mixed reviews."
]

# Analyze sentiment
sentiments = sentiment_analyzer.analyze_batch(financial_texts)

for text, sentiment in zip(financial_texts, sentiments):
    numeric_score = sentiment_analyzer.score_to_numeric(sentiment)
    print(f"\nText: {text}")
    print(f"Sentiment: {sentiment['label']} (score: {numeric_score:.3f})")

## Step 4: Generate Trading Signals

In [ ]:
# Create mock events and reactions
feature_extractor = ReactionFeatureExtractor(sentiment_analyzer)
signal_generator = ESGSignalGenerator()

signals_list = []

for ticker in tickers[:3]:
    event_date = datetime.strptime(start_date, '%Y-%m-%d') + timedelta(days=30)
    
    # Mock event
    event_result = {
        'has_event': True,
        'category': 'E',
        'confidence': 0.8,
        'sentiment': 'negative'
    }
    
    # Mock social media reaction
    mock_tweets = feature_extractor.create_mock_social_data(
        ticker, event_date, n_tweets=100, sentiment_bias='negative'
    )
    
    reaction_features = feature_extractor.extract_features(mock_tweets, event_date)
    
    # Generate signal
    signal = signal_generator.compute_signal(
        ticker=ticker,
        date=event_date,
        event_features=event_result,
        reaction_features=reaction_features
    )
    
    signals_list.append(signal)

signals_df = pd.DataFrame(signals_list)
print("\nGenerated Trading Signals:")
signals_df

## Step 5: Construct Portfolio

In [ ]:
# Construct portfolio
portfolio_constructor = PortfolioConstructor(strategy_type='long_short')
portfolio = portfolio_constructor.construct_portfolio(signals_df, method='quintile')

print("\nPortfolio Weights:")
print(portfolio)

# Portfolio statistics
stats = portfolio_constructor.get_portfolio_statistics(portfolio)
print("\nPortfolio Statistics:")
for key, value in stats.items():
    print(f"{key}: {value}")

## Step 6: Run Backtest

In [ ]:
# Run backtest
backtest_engine = BacktestEngine(
    prices=prices,
    initial_capital=1000000.0,
    commission_pct=0.0005,
    slippage_bps=3
)

results = backtest_engine.run(
    signals=portfolio,
    rebalance_freq='W',
    holding_period=10
)

print(f"\nFinal Portfolio Value: ${results.get_final_value():,.2f}")
print(f"Total Return: {results.get_total_return()*100:.2f}%")

## Step 7: Analyze Performance

In [ ]:
# Generate performance tear sheet
perf_analyzer = PerformanceAnalyzer(results)
perf_analyzer.print_tear_sheet()

## Step 8: Factor Analysis (Proving Alpha)

In [ ]:
# Load Fama-French factors
ff_factors = FamaFrenchFactors()
factors = ff_factors.load_ff_factors(start_date, end_date, frequency='daily')

# Run factor regression
factor_analysis = FactorAnalysis()
factor_analysis.load_factors(factors)
regression_results = factor_analysis.run_regression(results.returns_series)

# Print interpretation
print(factor_analysis.interpret_results(regression_results))

## Visualization (Optional)

In [ ]:
import matplotlib.pyplot as plt

# Plot equity curve
equity_curve = results.get_equity_curve()

if not equity_curve.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(equity_curve.index, equity_curve.values)
    plt.title('Portfolio Equity Curve')
    plt.xlabel('Date')
    plt.ylabel('Portfolio Value ($)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No equity curve data available")

## Conclusion

This notebook demonstrated the complete pipeline:
1. ✓ Data acquisition
2. ✓ ESG event detection
3. ✓ Sentiment analysis
4. ✓ Signal generation
5. ✓ Portfolio construction
6. ✓ Backtesting
7. ✓ Performance analysis
8. ✓ Factor analysis (proving alpha)

Next steps:
- Run with real SEC filings
- Add social media data (Twitter API)
- Optimize signal weights
- Implement walk-forward testing
- Paper trading before live deployment